In [29]:
import pandas as pd
import numpy as np

df_reservas = pd.read_excel('dados/reserva.xlsx', sheet_name='Reservas')
df_clientes = pd.read_excel('dados/reserva.xlsx', sheet_name='Clientes')
df_pagamentos = pd.read_excel('dados/reserva.xlsx', sheet_name='Pagamentos')

display(df_reservas.columns.to_list())
display(df_clientes.columns.to_list())
display(df_pagamentos.columns.to_list())

# Criar bases tratadas preservando os dados originais
preservar_reservas = df_reservas.copy()
preservar_pagamentos = df_pagamentos.copy()
preservar_clientes = df_clientes.copy()


# 1. auditoria e limpeza

antes_reserva = len(df_reservas['Reserva_ID'])
antes_clientes = len(df_clientes['Cliente_ID'])
antes_pagamentos = len(df_pagamentos['Pagamento_ID'])

# 1a. identificar valores e registros duplicados nas três tabelas

duplicada_reserva = df_reservas[df_reservas.duplicated(subset='Reserva_ID')]
duplicada_cliente = df_clientes[df_clientes.duplicated(subset='Cliente_ID')]
duplicada_pagamento = df_pagamentos[df_pagamentos.duplicated(subset='Pagamento_ID')]

display('Duplicadas em reservas', duplicada_reserva)
display('duplicada em clientes ', duplicada_cliente) 
display('duplicada em pagamentos', duplicada_pagamento)

df_reservas = df_reservas.drop_duplicates(
    subset=['Reserva_ID'],
    keep='first'
)

# unica tabela com limpeza reservas

display('Antes:', antes_reserva)
display('Depois: ', len(df_reservas['Reserva_ID']))







['Reserva_ID',
 'Cliente_ID',
 'Data_Entrada',
 'Canal',
 'Diarias',
 'Valor_R$',
 'Status']

['Cliente_ID', 'Nome', 'UF', 'Email']

['Pagamento_ID', 'Reserva_ID', 'Metodo', 'Valor_Pago_R$', 'Status_Pagamento']

'Duplicadas em reservas'

,Reserva_ID,Cliente_ID,Data_Entrada,Canal,Diarias,Valor_R$,Status
42,1012,212,2026-04-03,Instagram,2.0,690.0,Cancelada


...

,Pagamento_ID,Reserva_ID,Metodo,Valor_Pago_R$,Status_Pagamento


'Antes:'

43

'Depois: '

42

In [26]:
# 2. Integração

# Integrar reservas e clientes utilizando a chave adequada

df_relacao = pd.merge(
    df_reservas,
    df_clientes,
    on='Cliente_ID',
    how='left',
    validate='many_to_one',
    indicator=True
)

display(df_relacao.head())

pagamentos_reserva = df_pagamentos.groupby(
    ['Reserva_ID', 'Status_Pagamento'], as_index=False
)['Valor_Pago_R$'].sum()

display(pagamentos_reserva.head())

display(pagamentos_reserva.duplicated())

df_integra = pd.merge(
    df_relacao,
    pagamentos_reserva,
    on='Reserva_ID',
    how='left',
    validate='one_to_one',
    indicator=True
)

display(df_relacao.head())





,Reserva_ID,Cliente_ID,Data_Entrada,Canal,Diarias,Valor_R$,Status,Nome,UF,Email,_merge
0,1001,201,2026-01-05,WhatsApp,1.0,290.0,Confirmada,Ana,SC,cliente1@exemplo.com,both
1,1002,202,2026-01-13,Instagram,2.0,1130.0,Concluida,Bruno,RS,cliente2@exemplo.com,both
2,1003,203,2026-01-21,Booking,3.0,1530.0,Cancelada,Camila,SP,cliente3@exemplo.com,both
3,1004,204,2026-01-29,Telefone,4.0,1820.0,Confirmada,Diego,PR,NaN,both
4,1005,205,2026-02-06,NaN,5.0,2000.0,Concluida,Elisa,RJ,cliente5@exemplo.com,both


,Reserva_ID,Status_Pagamento,Valor_Pago_R$
0,1001,Pendente,290.0
1,1002,Pago,1130.0
2,1003,Pago,1530.0
3,1004,Pago,1820.0
4,1005,Pendente,2000.0


0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
20    False
21    False
22    False
23    False
24    False
25    False
26    False
27    False
28    False
29    False
30    False
31    False
32    False
33    False
34    False
35    False
36    False
37    False
38    False
39    False
dtype: bool

ValueError: Cannot use name of an existing column for indicator column